<a href="https://colab.research.google.com/github/Devadeth-cmyk/E-LMS/blob/main/fake_or_real.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Libraries

In [15]:
import numpy as np
import pandas as pd
import string
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [25]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [29]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

#Read Data

In [17]:
filepath = '/content/drive/MyDrive/AI ML/Datasets/Fake.csv'
df_fake  = pd.read_csv(filepath)
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


#EDA

In [18]:
df_fake.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB


In [19]:
# Missing value checking
print('Missing values in df_fake:')
display(df_fake.isnull().sum())

Missing values in df_fake:


,0
title,0
text,0
subject,0
date,0


##Duplicate check

In [20]:
# Duplicate value check
df_fake.duplicated().sum()

np.int64(3)

In [21]:
#Removing duplicates
df_fake.drop_duplicates(inplace=True)

In [22]:
# Duplicate value check
df_fake.duplicated().sum()

np.int64(0)

#Preprocessing

##Lowercasing

In [23]:
# Combine 'title' and 'text' columns for preprocessing
df_fake['full_text'] = df_fake['title'] + ' ' + df_fake['text']

# Function to remove punctuation and convert to lowercase
def clean_text(text):
    text = str(text).lower()
    text = ''.join([char for char in text if char not in string.punctuation]) # Remove punctuation
    return text

df_fake['cleaned_text'] = df_fake['full_text'].apply(clean_text)
display(df_fake[['full_text', 'cleaned_text']].head())

,full_text,cleaned_text
0,Donald Trump Sends Out Embarrassing New Year’...,donald trump sends out embarrassing new year’...
1,Drunk Bragging Trump Staffer Started Russian ...,drunk bragging trump staffer started russian ...
2,Sheriff David Clarke Becomes An Internet Joke...,sheriff david clarke becomes an internet joke...
3,Trump Is So Obsessed He Even Has Obama’s Name...,trump is so obsessed he even has obama’s name...
4,Pope Francis Just Called Out Donald Trump Dur...,pope francis just called out donald trump dur...


##Tokenization

In [31]:
# Tokenization
def tokenize_text(text):
    return nltk.word_tokenize(text)

df_fake['tokenized_text'] = df_fake['cleaned_text'].apply(tokenize_text)
display(df_fake[['cleaned_text', 'tokenized_text']].head())

,cleaned_text,tokenized_text
0,donald trump sends out embarrassing new year’...,"[donald, trump, sends, out, embarrassing, new,..."
1,drunk bragging trump staffer started russian ...,"[drunk, bragging, trump, staffer, started, rus..."
2,sheriff david clarke becomes an internet joke...,"[sheriff, david, clarke, becomes, an, internet..."
3,trump is so obsessed he even has obama’s name...,"[trump, is, so, obsessed, he, even, has, obama..."
4,pope francis just called out donald trump dur...,"[pope, francis, just, called, out, donald, tru..."


##Stop word removal

In [32]:
# Stopword Removal
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

df_fake['text_without_stopwords'] = df_fake['tokenized_text'].apply(remove_stopwords)
display(df_fake[['tokenized_text', 'text_without_stopwords']].head())

,tokenized_text,text_without_stopwords
0,"[donald, trump, sends, out, embarrassing, new,...","[donald, trump, sends, embarrassing, new, year..."
1,"[drunk, bragging, trump, staffer, started, rus...","[drunk, bragging, trump, staffer, started, rus..."
2,"[sheriff, david, clarke, becomes, an, internet...","[sheriff, david, clarke, becomes, internet, jo..."
3,"[trump, is, so, obsessed, he, even, has, obama...","[trump, obsessed, even, obama, ’, name, coded,..."
4,"[pope, francis, just, called, out, donald, tru...","[pope, francis, called, donald, trump, christm..."


##Lemmatization

In [33]:
# Lemmatization
lemmatizer = WordNetLemmatizer()

def lemmatize_text(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df_fake['lemmatized_text'] = df_fake['text_without_stopwords'].apply(lemmatize_text)
display(df_fake[['text_without_stopwords', 'lemmatized_text']].head())

,text_without_stopwords,lemmatized_text
0,"[donald, trump, sends, embarrassing, new, year...","[donald, trump, sends, embarrassing, new, year..."
1,"[drunk, bragging, trump, staffer, started, rus...","[drunk, bragging, trump, staffer, started, rus..."
2,"[sheriff, david, clarke, becomes, internet, jo...","[sheriff, david, clarke, becomes, internet, jo..."
3,"[trump, obsessed, even, obama, ’, name, coded,...","[trump, obsessed, even, obama, ’, name, coded,..."
4,"[pope, francis, called, donald, trump, christm...","[pope, francis, called, donald, trump, christm..."


In [34]:
# Join the lemmatized tokens back into a string
df_fake['processed_text'] = df_fake['lemmatized_text'].apply(lambda x: ' '.join(x))
display(df_fake[['full_text', 'processed_text']].head())

,full_text,processed_text
0,Donald Trump Sends Out Embarrassing New Year’...,donald trump sends embarrassing new year ’ eve...
1,Drunk Bragging Trump Staffer Started Russian ...,drunk bragging trump staffer started russian c...
2,Sheriff David Clarke Becomes An Internet Joke...,sheriff david clarke becomes internet joke thr...
3,Trump Is So Obsessed He Even Has Obama’s Name...,trump obsessed even obama ’ name coded website...
4,Pope Francis Just Called Out Donald Trump Dur...,pope francis called donald trump christmas spe...


#Model building

#Evaluation